In [63]:
import os
import sys
import pandas as pd
import json
import ast

In [64]:
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

In [65]:
current_dir = os.getcwd()

# 프로젝트 루트 디렉토리 찾기 (notebooks/preprocessing에서 두 단계 위로 이동)
root_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))
print(f"프로젝트 루트 디렉토리: {root_dir}")
sys.path.append(root_dir)


프로젝트 루트 디렉토리: /Users/hazel/Documents/map-search-agent


In [66]:
meta_data = pd.read_csv('step_1.csv')

In [67]:
meta_data.head(1)

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,smsText.includedText.value,smsText.includedTextSeparateSetting.textRange,includedData.value,additionalDataUsage.includedDataForSharingAndTethering.value,additionalDataUsage.includedMVoIP.value,dataQoS.appliedSpeed.value,seniorDataExceedLimit.availableToApply.value,generalDataExceedLimit.availableToApply.value,monthlyPrice.monthlyPrice.value,monthlyPrice.monthlyPriceWithoutVAT.value,monthlyPrice.monthlyPriceWithSelectableInstallment.value,monthlyPrice.billingMethod.value,optionData.dataOptionProvidingMethod,deductibleInfo.deductibilityForDisability.value,benefitOfData.dataOptionRefill.dataRefillAmount.value,benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value,benefitOfData.dataOptionGift.maximumShareAmount.value,benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value,managementInfo.productId.value,managementInfo.statusOfOperation.value,managementInfo.classifiedGroup.value,managementInfo.productName.value,managementInfo.productNameInEnglish.value,managementInfo.lineup.value,managementInfo.marketingKeyword.valueList,managementInfo.generation.valueList,managementInfo.mappedProductCode.productCode.valueList,managementInfo.productDescription.value,managementInfo.productSubscriptionCondition.value,salesInfo.netPrice.value,topupInfo.reChargeAvailability.availability.value,customerInfo.onboardingCustomer.ageRule,otherOnboardInfo.directPlanOnboard.value,otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value,otherOnboardInfo.tsupportFundOnboard.value,productRelation.signupPreTermination.productInformation.productList,productRelation.signupPreTermination.productInformation.groupList,productRelation.signupConcurrentTermination.productInformation.productList,productRelation.signupConcurrentTermination.productInformation.groupList,productRelation.terminationPreTermination.productInformation.productList,productRelation.terminationPreTermination.productInformation.groupList,productRelation.terminationConcurrentTermination.productInformation.productList,productRelation.terminationConcurrentTermination.productInformation.groupList,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value,productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList,productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits,optionData.optionDataName.value,optionData.selectionMethod.value,optionData.totalNumOfOptions.value,optionData.minNumOfOptionSelectable.value,optionData.maxNumOfOptionSelectable.value,deductibleInfo.additionalOfferForDisabilities.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value,productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value,customerInfo.onboardingCustomer.customerTypeRule.eligibility,customerInfo.onboardingCustomer.customerTypeRule.valueList,otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList,productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList,topupInfo.chargeAmount.minimumChargeAmount.value,topupInfo.chargeAmount.maximumChargeAmount.value,otherOnboardInfo.specialCustomerOnboard.isSoldier.value,otherOnboardInfo.duplicateNameOnboard.productGroup.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList,benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount,benefitOfVoiceCall.performRefill.voiceCallRefillRange.range,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService,productBenefitConditions.allOfferBenefits.benefitInfo
0,PA00000001,무제한,300분,기본제공,[],15GB,15GB,15GB,1Mbps,N,N,38000원,345

### 2-4) 정규식 : '~분, ~건' 형태의 필드를 숫자값만 남김
- voice.includedVoiceCall.value
- voice.includedVideoOrValueAddedCall.value
- smsText.includedText.value

In [ ]:
# 기존 요금제 agent에서 하는 식으로 하면 60분, 30분 이런게 다 0으로 되버림 
def str_to_num(x):
    if x == '무제한':
        return 99999
    elif x.endswith(('분', '건')):
        try:
            return int(x[:-1])
        except ValueError:
            return 0
    else:
        return 0

In [ ]:
meta_data['voice.includedVoiceCall.value'] = meta_data['voice.includedVoiceCall.value'].apply(str_to_num)
meta_data['voice.includedVideoOrValueAddedCall.value'] = meta_data['voice.includedVideoOrValueAddedCall.value'].apply(str_to_num)
meta_data['smsText.includedText.value'] = meta_data['smsText.includedText.value'].apply(str_to_num)

### 2-5) 정규식 : 데이터 용량 필드를 단위 맞춰서 숫자값만 남김
- includedData.value
- additionalDataUsage.includedDataForSharingAndTethering.value
- additionalDataUsage.includedMVoIP.value
- benefitOfData.dataOptionRefill.dataRefillAmount.value

In [78]:
def to_gb(x):
    if x == '무제한':
        return 99999.0
    elif x.endswith('GB'):
        try:
            return float(x.replace('GB', ''))
        except ValueError:
            return 0.0
    elif x.endswith('MB'):
        try:
            return float(x.replace('MB', '')) / 1024
        except ValueError:
            return 0.0
    else:
        return 0.0

In [ ]:
meta_data['includedData.value'] = meta_data['includedData.value'].apply(to_gb)
# 제공량이 없는 경우 0으로 채움
meta_data['additionalDataUsage.includedDataForSharingAndTethering.value'] = meta_data['additionalDataUsage.includedDataForSharingAndTethering.value'].fillna('0GB')
meta_data['additionalDataUsage.includedDataForSharingAndTethering.value'] = meta_data['additionalDataUsage.includedDataForSharingAndTethering.value'].apply(to_gb)
meta_data['additionalDataUsage.includedMVoIP.value'] = meta_data['additionalDataUsage.includedMVoIP.value'].fillna('0GB')
meta_data['additionalDataUsage.includedMVoIP.value'] = meta_data['additionalDataUsage.includedMVoIP.value'].apply(to_gb)
meta_data['benefitOfData.dataOptionRefill.dataRefillAmount.value'] = meta_data['benefitOfData.dataOptionRefill.dataRefillAmount.value'].fillna('무제한')
meta_data['benefitOfData.dataOptionRefill.dataRefillAmount.value'] = meta_data['benefitOfData.dataOptionRefill.dataRefillAmount.value'].apply(to_gb)

In [ ]:
##### 진아님께 질문했는데 결과 오면 채워넣어야 함. 일반 정책으로 2GB가 기본값인지 문의했음
meta_data[['benefitOfData.dataOptionGift.maximumShareAmount.value', 'includedData.value']]

,benefitOfData.dataOptionGift.maximumShareAmount.value,includedData.value
0,2GB,15.000000
1,2GB,1.800000
2,2GB,5.000000
3,2GB,100.000000
4,2GB,200.000000
5,2GB,24.000000
6,2GB,11.000000
7,2GB,110.000000
8,2GB,250.000000
9,2GB,99999.000000


### 2-6) 정규식 : 데이터 속도 필드를 단위 맞춰서 숫자값만 남김
- dataQoS.appliedSpeed.value

In [ ]:
def to_mbps(x):
    x = x.strip()
    if x.lower().endswith('mbps'):
        return float(x.lower().replace('mbps', ''))
    elif x.lower().endswith('kbps'):
        return float(x.lower().replace('kbps', '')) / 1000
    else:
        return 0.0

# 속도 값이 없는 경우 0으로 채움
meta_data['dataQoS.appliedSpeed.value'] = meta_data['dataQoS.appliedSpeed.value'].fillna('0Mbps')
meta_data['dataQoS.appliedSpeed.value'] = meta_data['dataQoS.appliedSpeed.value'].apply(to_mbps)

### 2-7) 정규식 : 금액 관련 필드를 단위 맞춰 숫자값만 남김
- monthlyPrice.monthlyPrice.value
- monthlyPrice.monthlyPriceWithoutVAT.value
- monthlyPrice.monthlyPriceWithSelectableInstallment.value

In [94]:
def remove_won(x):
    # '원' 제거, 쉼표 제거, 공백 제거 후 숫자로 변환
    return int(x.replace('원', '').replace(',', '').strip())

In [ ]:
meta_data['monthlyPrice.monthlyPrice.value'] = meta_data['monthlyPrice.monthlyPrice.value'].apply(remove_won)
meta_data['monthlyPrice.monthlyPriceWithoutVAT.value'] = meta_data['monthlyPrice.monthlyPriceWithoutVAT.value'].apply(remove_won)
meta_data['monthlyPrice.monthlyPriceWithSelectableInstallment.value'] = meta_data['monthlyPrice.monthlyPriceWithSelectableInstallment.value'].apply(remove_won)

In [152]:
# salesInfo.netPrice.value 의 나머지 케이스 확인해보니 monthlyPrice * 0.9901 한 후 소수점 이하를 버림 하였음. 동일하게 처리 
meta_data.loc[meta_data['salesInfo.netPrice.value'].isnull(), 'salesInfo.netPrice.value'] = str(int(round(meta_data[meta_data['salesInfo.netPrice.value'].isnull()]['monthlyPrice.monthlyPrice.value'].values[0]*0.9901, -1)))+'원'

In [154]:
meta_data.to_csv('step_2.csv',index=False, encoding='utf-8-sig')